# Missing Value Analysis

This notebook analyzes missing values across the raw Instacart datasets.

The analysis identifies the location and extent of missing data and distinguishes expected structural missingness from potential data-quality issues before data cleaning begins.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw")

DATA_FILES = {
    "orders": "orders.csv",
    "order_products_prior": "order_products__prior.csv",
    "order_products_train": "order_products__train.csv",
    "products": "products.csv",
    "aisles": "aisles.csv",
    "departments": "departments.csv",
}

## Analyze Missing Values

Missing values are calculated for every column across all raw datasets.

Large transaction tables are processed in chunks to avoid unnecessary memory consumption.

In [3]:
def calculate_missing_values(path, chunksize=200_000):
    missing = {}

    for chunk in pd.read_csv(path, chunksize=chunksize):
        for column, count in chunk.isna().sum().items():
            missing[column] = missing.get(column, 0) + int(count)

    return pd.Series(missing).sort_values(ascending=False)


missing_results = {}

for name, filename in DATA_FILES.items():
    missing_results[name] = calculate_missing_values(
        DATA_DIR / filename
    )

    print(f"\n{name.upper()}")
    print("-" * 60)
    print(missing_results[name])


ORDERS
------------------------------------------------------------
days_since_prior_order    206209
user_id                        0
order_id                       0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
dtype: int64

ORDER_PRODUCTS_PRIOR
------------------------------------------------------------
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

ORDER_PRODUCTS_TRAIN
------------------------------------------------------------
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

PRODUCTS
------------------------------------------------------------
product_id       0
product_name     0
aisle_id         0
department_id    0
dtype: int64

AISLES
------------------------------------------------------------
aisle_id    0
aisle       0
dtype: int64

DEPARTMENTS
----------------------------

In [4]:
missing_summary = pd.DataFrame(missing_results).fillna(0).astype(int)

missing_summary

,orders,order_products_prior,order_products_train,products,aisles,departments
add_to_cart_order,0,0,0,0,0,0
aisle,0,0,0,0,0,0
aisle_id,0,0,0,0,0,0
days_since_prior_order,206209,0,0,0,0,0
department,0,0,0,0,0,0
department_id,0,0,0,0,0,0
eval_set,0,0,0,0,0,0
order_dow,0,0,0,0,0,0
order_hour_of_day,0,0,0,0,0,0
order_id,0,0,0,0,0,0


In [5]:
non_zero_missing = {}

for name, result in missing_results.items():
    values = result[result > 0]

    if not values.empty:
        non_zero_missing[name] = values

print("=== COLUMNS WITH MISSING VALUES ===")

for name, values in non_zero_missing.items():
    print(f"\n{name.upper()}")
    print("-" * 60)
    print(values)

=== COLUMNS WITH MISSING VALUES ===

ORDERS
------------------------------------------------------------
days_since_prior_order    206209
dtype: int64


In [6]:
orders = pd.read_csv(
    DATA_DIR / DATA_FILES["orders"],
    usecols=["user_id", "order_number", "days_since_prior_order"]
)

missing_first_order = orders[
    orders["days_since_prior_order"].isna()
]

print("Missing days_since_prior_order:",
      len(missing_first_order))

print(
    "Missing values with order_number = 1:",
    (missing_first_order["order_number"] == 1).sum()
)

print(
    "Missing values with order_number > 1:",
    (missing_first_order["order_number"] > 1).sum()
)

Missing days_since_prior_order: 206209
Missing values with order_number = 1: 206209
Missing values with order_number > 1: 0


In [7]:
missing_analysis = pd.DataFrame({
    "Dataset": ["orders"],
    "Column": ["days_since_prior_order"],
    "Missing Values": [
        orders["days_since_prior_order"].isna().sum()
    ],
    "First Orders": [
        (
            orders.loc[
                orders["days_since_prior_order"].isna(),
                "order_number"
            ] == 1
        ).sum()
    ],
    "Non_First_Orders": [
        (
            orders.loc[
                orders["days_since_prior_order"].isna(),
                "order_number"
            ] > 1
        ).sum()
    ]
})

missing_analysis

,Dataset,Column,Missing Values,First Orders,Non_First_Orders
0,orders,days_since_prior_order,206209,206209,0
